- Dataset size: 5088300 (101766 x 50)

- All "?" values were replaced by Nan
- Percentage of NaN values per column:
    weight               96.86
    max_glu_serum        94.75
    A1Cresult            83.28
    medical_specialty    49.08
    payer_code           39.56
    race                  2.23
    diag_3                1.40
    diag_2                0.35
    diag_1                0.02

- "Readmitted" column:
    "NO": 53.91 %
    ">30": 34.92 %
    "<30": 11.12 %

- There are 30,248 patient_id lines that are repeated. The distinct patient_nbr values are 71,158.

- The following columns are useless for the training:
    "weight"
    "encounter_id"
    "patient_nbr" (after removing duplicates)
    "max_glu_serum"
    "A1Cresult"
    "medical_specialty"
    "payer_code"

In [2]:
#1.
import pandas as pd, numpy as np, seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
#2. dataframe with relative path to csv file
df = pd.read_csv("../data/diabetic_data.csv", na_values = "?")
print(df.head(10))

In [4]:
#3. removes duplicates based on patient_nbr column
df.drop_duplicates(subset="patient_nbr", inplace = True)

In [ ]:
perc = (df.isnull().sum() / df.shape[0]) * 100
perc[perc > 0].sort_values(ascending = False).round(2)

In [ ]:
#size of distict values in column "readmitted"
df["readmitted"].value_counts()

In [ ]:
#percentage of each readmission category using value_counts with normalize
df["readmitted"].value_counts(normalize=True)*100

In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
sns.countplot(x = "readmitted", data = df)


In [ ]:
#4. remove columns with many missing values
df = df.drop(columns = [
    "weight",
    "encounter_id",
    "patient_nbr",
    "max_glu_serum",
    "A1Cresult",
    "medical_specialty",
    "payer_code"])

In [ ]:
df.isnull().sum()

In [7]:
#5. remove lines where at least one of (diag_1/diag_2/dig_3) is empty
df.dropna(axis=0, subset= ["diag_1", "diag_2", "diag_3"], inplace = True)

In [17]:
#6. classify the positive class
df["readmitted"] = df["readmitted"].replace({"NO": 0, ">30": 0, "<30": 1})
df["readmitted"] =df["readmitted"].astype(int)
df["race"] = df["race"].fillna("Unknown", inplace = False)

In [ ]:
#8. create the classes

  #creation of OrdinalEncoder
ode = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1)

#categorical columns
cat_cols = X_train.select_dtypes(exclude = 'int64').columns

#encoding in train and test set
X_train[cat_cols] = ode.fit_transform(X_train[cat_cols])
X_test[cat_cols] = ode.transform(X_test[cat_cols])

In [ ]:
#9. Dataset spliting
Y = df["readmitted"]
X = df.drop(columns = ["readmitted"])

In [ ]:
#10. remove patients that expired or went to hospice
df=df[~df['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])]

In [24]:
#11. fill "unknown" where race value is missing
df["race"] = df["race"].fillna("Unknown")

In [25]:
#12. dataset splitting
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state = 18, test_size = 0.25, stratify = Y)

In [ ]:
print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)


In [ ]:
print((len(Y_train) - sum(Y_train))/sum(Y_train))

52673.0


In [ ]:
print(df['discharge_disposition_id'].value_counts())

In [6]:
#packages imported for preprocessing
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split


#function that returns the preprocessed data (X_train, X_test, Y_train, Y_test) for model training and evaluation
def data_preprocessing():


    #dataframe with relative path to csv file
    df = pd.read_csv("../data/diabetic_data.csv", na_values = "?", low_memory = False)

    #removes duplicates based on patient_nbr column
    df.drop_duplicates(subset="patient_nbr", inplace = True)

    #removal of rows where any of the diagnosis columns have missing values
    df.dropna(axis=0, subset= ["diag_1", "diag_2", "diag_3"], inplace = True)

    #remove patients that expired or went to hospice
    df=df[~df['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])]

    #removal of columns with many missing values and columns that are not useful for prediction
    df = df.drop(columns = [
        "weight",
        "encounter_id",
        "patient_nbr",
        "max_glu_serum",
        "A1Cresult",
        "medical_specialty",
        "payer_code"])

    #classify the target variable into binary classes
    df["readmitted"] = df["readmitted"].replace({"NO": 0, ">30": 0, "<30": 1})
    df["readmitted"] =df["readmitted"].astype(int)

    #filling missing values with unknown for the race column
    df["race"] = df["race"].fillna("Unknown")


    #dataset spliting into features and target variable
    Y = df["readmitted"]
    X = df.drop(columns = ["readmitted"])

    def map_icd9(code):

        if pd.isna(code):
            return "Other"
        integer_part = code.split('.')[0]
        try:
            n = int(integer_part)
        except ValueError:
            return "Other"
    
        if 390 <= n <= 459 or n == 785:
            return "Circulatory"
        elif 460 <= n <= 519 or n == 786:
            return "Respiratory"
        elif n == 250:
            return "Diabetes"
        elif 520 <= n <= 579 or n == 787:
            return 'Digestive'
        elif 800 <= n <= 999:
            return 'Injury'
        elif 710 <= n <= 739:
            return 'Musculoskeletal'
        elif 580 <= n <= 629 or n == 788:
            return 'Genitourinary'
        elif 140 <= n <= 239:
            return 'Neoplasms'
        else:
            return "Other"
        
    X['diag_1'] = X['diag_1'].apply(map_icd9)
    X['diag_2'] = X['diag_2'].apply(map_icd9)
    X['diag_3'] = X['diag_3'].apply(map_icd9)

#     #dataset spliting into training and testing sets
#     X_train, X_test, Y_train, Y_test = train_test_split(
#         X,
#         Y, 
#         random_state = 18,
#         test_size = 0.25,
#         stratify = Y)
    
    
#     #creation of OrdinalEncoder
#     ode = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1)

#     #categorical columns
#     cat_cols = X_train.select_dtypes(include = ['object', 'str']).columns

#     #encoding in train and test set
#     X_train[cat_cols] = ode.fit_transform(X_train[cat_cols])
#     X_test[cat_cols] = ode.transform(X_test[cat_cols])

    
#     return X_train, X_test, Y_train, Y_test
    return X
    


if __name__ == "__main__":
#     X_train, X_test, Y_train, Y_test = data_preprocessing()
    X = data_preprocessing()
    


In [7]:
X.dtypes

race                          str
gender                        str
age                           str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
acetohexamide                 str
glipizide                     str
glyburide                     str
tolbutamide                   str
pioglitazone                  str
rosiglitazone                 str
acarbose                      str
miglitol      